# BDC 2026 — Adaptasi k-NN dari ConvNeXt V2-Base

**Basis**: adaptasi dari laporan dosen "Candidate 27" -- bedanya, sumber angka k-NN di sini bukan embedding encoder frozen (SO400M, 1.152 dimensi), tapi **probabilitas softmax ConvNeXt V2-Base yang sudah di-fine-tune** (3 dimensi: Recyclable/Electronic/Organic).

**Semua pencarian parameter (k, mode k-NN, alpha campuran) di notebook ini dilakukan murni dari OOF** (`oof_predictions_wide.csv`) -- **belum menyentuh** `solution.csv` sama sekali. Prinsip ini dipegang ketat karena OOF (~26.527 sampel) jauh lebih tahan noise dibanding label manual test (~1.458 sampel), dan tidak "membocorkan" jawaban test ke proses pemilihan parameter.

**Input**: `oof_predictions_wide.csv` dan `test_predictions_convnextv2_base_wide.csv` dari notebook sebelumnya.
**Output**: `knn_config.json` -- parameter terbaik untuk dipakai di Notebook 3.

In [ ]:
# 1. IMPORT & LOAD DATA
import json
import numpy as np
import pandas as pd
from pathlib import Path
from sklearn.neighbors import NearestNeighbors
from sklearn.metrics import f1_score

# Path ABSOLUT -- konsisten dengan notebook 1 & 3, tidak bergantung cwd kernel.
PROJECT_ROOT = Path("C:/Users/MyPC PRO/Downloads/BDC2026")
DATA_DIR = PROJECT_ROOT / "clean_dataset_v3"
OUTPUT_DIR = PROJECT_ROOT / "files (1)" / "outputs"   # kanonik -- hasil lengkap ada di sini

CLASSES = ["0_Recyclable", "1_Electronic", "2_Organic"]
LABEL_MAPPING = {c: i for i, c in enumerate(CLASSES)}
CLASS_SHORT = ["recyclable", "electronic", "organic"]
MODEL_KEY = "convnext"
PROB_COLS = [f"{MODEL_KEY}_prob_{c}" for c in CLASS_SHORT]

oof_df = pd.read_csv(OUTPUT_DIR / "oof_predictions_wide.csv")
test_df = pd.read_csv(OUTPUT_DIR / "test_predictions_convnextv2_base_wide.csv")

assert oof_df.isna().sum().sum() == 0, "Ada NaN di OOF!"
assert all(c in oof_df.columns for c in PROB_COLS), f"Kolom {PROB_COLS} tidak lengkap di OOF"

X_oof = oof_df[PROB_COLS].values
y_oof = oof_df["true_label"].map(LABEL_MAPPING).values
X_test = test_df[PROB_COLS].values
test_ids = test_df["id"].values

print(f"OOF   : X={X_oof.shape}, y={y_oof.shape}")
print(f"Test  : X={X_test.shape}")

baseline_oof_f1 = f1_score(y_oof, np.argmax(X_oof, axis=1), average="macro")
print(f"\nBaseline OOF Macro-F1 (argmax ConvNeXt langsung, tanpa k-NN): {baseline_oof_f1:.4f}")

In [ ]:
# 2. FUNGSI k-NN -- exclude-self untuk OOF (leave-one-out), tanpa exclude untuk query ke test
def query_knn_exclude_self(nn_model, X, k):
    """Query k tetangga TERDEKAT dari X terhadap dirinya sendiri (index == data), dengan
    baris query itu sendiri (distance=0) dibuang -- supaya evaluasi OOF tidak bocor / trivial."""
    distances, indices = nn_model.kneighbors(X, n_neighbors=k + 1)
    n = X.shape[0]
    out_idx = np.zeros((n, k), dtype=int)
    out_dist = np.zeros((n, k))
    for i in range(n):
        mask = indices[i] != i
        sel = np.where(mask)[0][:k] if mask.sum() >= k else np.arange(min(k, len(indices[i])))
        out_idx[i] = indices[i][sel]
        out_dist[i] = distances[i][sel]
    return out_dist, out_idx

def knn_vote_probs(neighbor_idx, neighbor_dist, y_ref, num_classes, weighted):
    neighbor_labels = y_ref[neighbor_idx]                     # shape (n, k)
    w = 1.0 / (neighbor_dist + 1e-6) if weighted else np.ones_like(neighbor_dist)
    probs = np.zeros((neighbor_idx.shape[0], num_classes))
    for c in range(num_classes):
        probs[:, c] = np.sum(w * (neighbor_labels == c), axis=1)
    probs = probs / (probs.sum(axis=1, keepdims=True) + 1e-12)
    return probs

print("Fungsi k-NN siap.")

In [ ]:
# 3. GRID SEARCH -- k in [5,10,15,20,25], mode in [uniform, weighted], alpha in linspace(0,1,21)
# SEMUA dievaluasi terhadap y_oof (leave-one-out) -- tidak pernah menyentuh solution.csv
K_CANDIDATES = [5, 10, 15, 20, 25]
ALPHA_CANDIDATES = np.linspace(0.0, 1.0, 21)
num_classes = len(CLASSES)

nn_index_oof = NearestNeighbors(n_neighbors=max(K_CANDIDATES) + 1, algorithm="auto")
nn_index_oof.fit(X_oof)

results = []
knn_probs_cache = {}   # (k, weighted) -> probs, supaya tidak query ulang tiap alpha

for k in K_CANDIDATES:
    for weighted in [False, True]:
        dist, idx = query_knn_exclude_self(nn_index_oof, X_oof, k)
        knn_probs = knn_vote_probs(idx, dist, y_oof, num_classes, weighted)
        knn_probs_cache[(k, weighted)] = knn_probs

        for alpha in ALPHA_CANDIDATES:
            blended = alpha * X_oof + (1 - alpha) * knn_probs
            f1 = f1_score(y_oof, np.argmax(blended, axis=1), average="macro")
            results.append({"k": k, "weighted": weighted, "alpha": round(float(alpha), 3), "oof_f1": f1})

results_df = pd.DataFrame(results).sort_values("oof_f1", ascending=False).reset_index(drop=True)
print("Top 10 kombinasi terbaik (dari OOF):")
print(results_df.head(10).to_string(index=False))

In [ ]:
# 4. PILIH KONFIGURASI TERBAIK + BANDINGKAN DENGAN BASELINE
best_row = results_df.iloc[0]
best_k = int(best_row["k"])
best_weighted = bool(best_row["weighted"])
best_alpha = float(best_row["alpha"])
best_oof_f1 = float(best_row["oof_f1"])

pure_knn_f1 = results_df[(results_df["k"] == best_k) & (results_df["weighted"] == best_weighted) & (results_df["alpha"] == 0.0)]["oof_f1"].values[0]

print("=" * 60)
print("KONFIGURASI TERBAIK (dari OOF)")
print("=" * 60)
print(f"k              : {best_k}")
print(f"mode           : {'weighted (1/jarak)' if best_weighted else 'uniform'}")
print(f"alpha          : {best_alpha:.2f}  (bobot ConvNeXt murni; sisanya (1-alpha) bobot k-NN)")
print(f"OOF Macro-F1   : {best_oof_f1:.4f}")
print(f"\nPerbandingan:")
print(f"  ConvNeXt murni (alpha=1.0)      : {baseline_oof_f1:.4f}")
print(f"  k-NN murni (alpha=0.0, k={best_k}) : {pure_knn_f1:.4f}")
print(f"  Blended terbaik (alpha={best_alpha:.2f})   : {best_oof_f1:.4f}")

improvement = best_oof_f1 - baseline_oof_f1
print(f"\nPeningkatan vs ConvNeXt murni: {improvement:+.4f}")
if improvement < 0.001:
    print("[CATATAN] Peningkatan sangat kecil/nihil -- k-NN mungkin tidak menambah nilai signifikan di dataset ini.")
    print("          Tetap lanjutkan ke Notebook 3, tapi pertimbangkan submission ConvNeXt murni sebagai cadangan.")

In [ ]:
# 5. SIMPAN KONFIGURASI UNTUK NOTEBOOK 3
knn_config = {
    "model_key": MODEL_KEY,
    "prob_cols": PROB_COLS,
    "best_k": best_k,
    "best_weighted": best_weighted,
    "best_alpha": best_alpha,
    "oof_f1_blended": best_oof_f1,
    "oof_f1_baseline_convnext": float(baseline_oof_f1),
    "oof_f1_pure_knn": float(pure_knn_f1),
}
config_path = OUTPUT_DIR / "knn_config.json"
with open(config_path, "w", encoding="utf-8") as f:
    json.dump(knn_config, f, indent=2)

results_df.to_csv(OUTPUT_DIR / "knn_grid_search_results.csv", index=False)

print(f"[SAVED] {config_path}")
print(json.dumps(knn_config, indent=2))

## Selesai -- Lanjut ke Notebook Berikutnya

`knn_config.json` berisi parameter (k, mode, alpha) yang **murni dipilih dari OOF**, siap dipakai untuk membuat prediksi Data Uji di Notebook 3 (Tahap C -- versi "Terkunci").

Lanjut ke: **`BDC2026_3TahapLanjutan_LimaKesepakatanKonflik.ipynb`**